# Notebook 5 — Peer Comparison Engine
**Purpose:** For any given company, find its top 5 financial peers using cosine similarity.
Validate: TCS peers should be INFY, WIPRO — not ADANIPOWER.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, sqlite3, os, sys
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
sys.path.insert(0, os.path.abspath('..'))
sns.set_theme(style='darkgrid'); plt.rcParams['figure.figsize'] = (14, 6)

conn = sqlite3.connect(os.path.join('..', 'db.sqlite3'))
companies = pd.read_sql('SELECT c.*, s.sector_name FROM dim_company c LEFT JOIN dim_sector s ON c.sector_id=s.sector_id', conn)
years = pd.read_sql('SELECT * FROM dim_year ORDER BY sort_order', conn)
pl = pd.read_sql('SELECT * FROM fact_profit_loss', conn).merge(years[['year_id','sort_order']], on='year_id')
bs = pd.read_sql('SELECT * FROM fact_balance_sheet', conn).merge(years[['year_id','sort_order']], on='year_id')
cf = pd.read_sql('SELECT * FROM fact_cash_flow', conn).merge(years[['year_id','sort_order']], on='year_id')
analysis = pd.read_sql('SELECT * FROM fact_analysis', conn)

## Step 1: Build Feature Matrix

In [ ]:
latest_pl = pl.sort_values('sort_order').groupby('company_id').last().reset_index()
latest_bs = bs.sort_values('sort_order').groupby('company_id').last().reset_index()
latest_cf = cf.sort_values('sort_order').groupby('company_id').last().reset_index()
growth_3y = analysis[analysis['period_label']=='3Y'][['company_id','compounded_sales_growth_pct']]

feat = companies[['symbol','company_name','sector_name','roe_pct']].copy()
feat = feat.merge(latest_pl[['company_id','opm_pct','dividend_payout_pct','net_profit','sales','eps']], left_on='symbol', right_on='company_id', how='left')
feat = feat.merge(latest_bs[['company_id','debt_to_equity']], left_on='symbol', right_on='company_id', how='left', suffixes=('','_bs'))
feat = feat.merge(latest_cf[['company_id','operating_activity']], left_on='symbol', right_on='company_id', how='left', suffixes=('','_cf'))
feat = feat.merge(growth_3y.rename(columns={'compounded_sales_growth_pct':'growth_3y'}), left_on='symbol', right_on='company_id', how='left', suffixes=('','_an'))
feat['cash_conversion'] = np.where(feat['net_profit']>0, feat['operating_activity']/feat['net_profit'], 0)

FEATURE_COLS = ['opm_pct','roe_pct','growth_3y','debt_to_equity','cash_conversion','dividend_payout_pct']
feat_clean = feat.dropna(subset=FEATURE_COLS).copy().reset_index(drop=True)

scaler = StandardScaler()
X = scaler.fit_transform(feat_clean[FEATURE_COLS])
print(f'Feature matrix: {X.shape[0]} companies × {X.shape[1]} features')

## Step 2: Compute Cosine Similarity

In [ ]:
sim_matrix = cosine_similarity(X)
sim_df = pd.DataFrame(sim_matrix, index=feat_clean['symbol'], columns=feat_clean['symbol'])

def get_peers(symbol, n=5):
    if symbol not in sim_df.index:
        return f'{symbol} not found'
    peers = sim_df[symbol].drop(symbol).nlargest(n)
    result = []
    for peer_sym, score in peers.items():
        peer_row = feat_clean[feat_clean['symbol']==peer_sym].iloc[0]
        result.append({'peer': peer_sym, 'name': peer_row['company_name'],
                       'sector': peer_row['sector_name'], 'similarity': round(score, 4)})
    return pd.DataFrame(result)

print('=== Top 5 Peers for TCS ===')
print(get_peers('TCS').to_string(index=False))

## Step 3: Validate Peer Mappings

In [ ]:
test_companies = ['TCS', 'HDFCBANK', 'RELIANCE', 'ADANIPOWER', 'WIPRO']
for sym in test_companies:
    print(f'\n=== Peers for {sym} ===')
    peers = get_peers(sym)
    if isinstance(peers, str):
        print(peers)
    else:
        print(peers.to_string(index=False))

## Step 4: Similarity Heatmap (Top 20 Companies)

In [ ]:
top20 = feat_clean.nlargest(20, 'sales')['symbol'].tolist()
top20_sim = sim_df.loc[top20, top20]
plt.figure(figsize=(14, 12))
sns.heatmap(top20_sim, annot=True, fmt='.2f', cmap='RdYlGn', square=True, vmin=-1, vmax=1)
plt.title('Cosine Similarity — Top 20 Companies by Revenue', fontweight='bold', fontsize=14)
plt.tight_layout(); plt.show()

## Step 5: Export Peer Mapping

In [ ]:
peer_rows = []
for sym in feat_clean['symbol']:
    peers = get_peers(sym, n=5)
    if isinstance(peers, pd.DataFrame):
        for _, p in peers.iterrows():
            peer_rows.append({'company': sym, 'peer': p['peer'],
                              'peer_name': p['name'], 'peer_sector': p['sector'],
                              'similarity': p['similarity']})

peer_df = pd.DataFrame(peer_rows)
peer_df.to_csv('../data/peer_mapping.csv', index=False)
print(f'✅ Exported {len(peer_df)} peer mappings to data/peer_mapping.csv')
conn.close()